In [23]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("../data/raw/Elimited_messy_sales_data.csv")
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,530331,22961,JUMBO BAG RED RETROSPOT,37,8/28/2025 17:08,2.08,17013.0,United Kingdom
1,530359,22457,CREAM CUPID HEARTS COAT HANGER,51,10/13/2025 11:48,2.75,13181.0,United Kingdom
2,530400,21212,PACK OF 72 RETROSPOT CAKE CASES,11,12/18/2025 16:59,0.55,17763.0,United Kingdom
3,530024,22720,POPCORN HOLDER,33,9/17/2025 3:55,0.85,17122.0,United Kingdom
4,C530303,22961,JUMBO BAG RED RETROSPOT,-2,8/29/2025 1:40,2.08,15169.0,United Kingdom


In [24]:
df.shape

(528, 8)

In [25]:
df.columns

Index(['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate',
       'UnitPrice', 'CustomerID', 'Country'],
      dtype='str')

In [26]:
df.dtypes

InvoiceNo          str
StockCode          str
Description        str
Quantity         int64
InvoiceDate        str
UnitPrice      float64
CustomerID     float64
Country            str
dtype: object

In [27]:
df.isnull().sum()

InvoiceNo       0
StockCode       0
Description     7
Quantity        0
InvoiceDate     0
UnitPrice       0
CustomerID     10
Country         0
dtype: int64

In [28]:
df.duplicated().sum()

np.int64(8)

In [29]:
df[["Quantity", "UnitPrice"]].describe()

,Quantity,UnitPrice
count,528.000000,528.000000
mean,70.212121,3.515265
std,583.141382,3.938864
min,-51.000000,0.000000
25%,14.000000,1.650000
50%,28.000000,2.080000
75%,42.000000,2.950000
max,9999.000000,18.000000


In [30]:
(df["Quantity"] < 0).sum()

np.int64(13)

In [31]:
(df["Quantity"] == 0).sum()

np.int64(4)

In [32]:
(df["UnitPrice"] == 0).sum()

np.int64(5)

In [33]:
df["InvoiceNo"].astype(str).str.startswith("C").sum()

np.int64(12)

In [34]:
df["Country"].unique()

<ArrowStringArray>
['United Kingdom',           'EIRE',        'GERMANY',       'PORTUGAL',
          'SPAIN',      'Australia',       'Portugal',           'Eire',
        'Belgium',          'Spain',        'BELGIUM',    'Switzerland',
         'France', 'UNITED KINGDOM',    'Netherlands',        'Germany',
         'FRANCE']
Length: 17, dtype: str

## Part 1: Data Understanding

- Dataset has 528 rows and 8 columns.
- Columns: InvoiceNo, StockCode, Description, Quantity, InvoiceDate, UnitPrice, CustomerID, Country.
- InvoiceDate is stored as text, not a real date, this needs to be converted.
- CustomerID is stored as a decimal number because it has missing values.
- Description is missing in 7 rows.
- CustomerID is missing in 10 rows.
- There are 8 exact duplicate rows.
- 13 rows have negative Quantity, but only 12 InvoiceNo values start with "C" (the usual marker for cancelled orders). This means one negative quantity row does not follow the expected pattern and needs a closer look.
- 4 rows have Quantity equal to 0, which is not a valid sale.
- 5 rows have UnitPrice equal to 0, which is also not valid for a real transaction.
- The highest Quantity value is 9999, which is far above the typical range and looks like an error rather than a real order.
- Country names are inconsistent, some appear in normal case (e.g. "Germany") and others in all uppercase (e.g. "GERMANY"), which would cause the same country to be counted separately during analysis.

## Part 2: Data Cleaning

FixingInvoiceDate (convert to real date)

In [35]:
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])
df["InvoiceDate"].dtype

dtype('<M8[us]')

Removing duplicates

There are 8 exact duplicate rows. Since these represent the same transaction recorded twice, we will remove the extra copies and keep only one of each.

In [36]:
df = df.drop_duplicates()
df.shape

(520, 8)

Handling missing Description

7 rows are missing a Description. Since we cannot guess the product name, and it is only a small number of rows, we will drop these rows rather than risk incorrect analysis.

In [37]:
df = df.dropna(subset=["Description"])
df.shape

(513, 8)

Handling missing CustomerID

10 rows are missing a CustomerID. These sales are still valid, so instead of removing them, we will label the missing values as "Unknown". This keeps them in overall revenue calculations, while making it clear they cannot be included in customer-specific analysis.

In [38]:
df["CustomerID"] = df["CustomerID"].fillna("Unknown")
df["CustomerID"].isnull().sum()

np.int64(0)

Handling cancelled transactions

13 rows have a negative Quantity, which represents cancelled orders. These are not real sales, so we will remove them from the dataset used for revenue analysis.

In [39]:
df = df[df["Quantity"] > 0]
df.shape

(496, 8)

Handling zero UnitPrice

5 rows have a UnitPrice of 0. Instead of removing these rows, we checked whether the same product appears elsewhere in the data with a valid price. All 4 affected products (22383, 84406B, 22698, 21755) have a consistent price in other rows, so we will use that price to fill in the missing ones instead of losing the data.

In [40]:
zero_price_products = df[df["UnitPrice"] == 0]["StockCode"].unique()
zero_price_products

<ArrowStringArray>
['22383', '84406B', '22698', '21755']
Length: 4, dtype: str

In [41]:
df[df["StockCode"].isin(zero_price_products) & (df["UnitPrice"] > 0)][["StockCode", "Description", "UnitPrice"]]

,StockCode,Description,UnitPrice
6,22383,WHITE HANGING HEART T-LIGHT HOLDER,2.95
13,22698,CREAM CUPID HEARTS COAT HANGER,2.75
16,22698,CREAM CUPID HEARTS COAT HANGER,2.75
19,21755,SET 7 BABUSHKA NESTING BOXES,8.50
20,22698,CREAM CUPID HEARTS COAT HANGER,2.75
...,...,...,...
500,84406B,CREAM CUPID HEARTS COAT HANGER,2.75
502,22383,WHITE HANGING HEART T-LIGHT HOLDER,2.95
507,22383,WHITE HANGING HEART T-LIGHT HOLDER,2.95
519,22383,WHITE HANGING HEART T-LIGHT HOLDER,2.95


In [42]:
price_lookup = df[df["UnitPrice"] > 0].groupby("StockCode")["UnitPrice"].agg(lambda x: x.mode()[0])

zero_price_mask = df["UnitPrice"] == 0
df.loc[zero_price_mask, "UnitPrice"] = df.loc[zero_price_mask, "StockCode"].map(price_lookup)

(df["UnitPrice"] == 0).sum()

np.int64(0)

Investigate the unusual values

In [43]:
df[df["Quantity"] > 500]

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
79,530309,22383,WHITE HANGING HEART T-LIGHT HOLDER,5000,2025-09-21 12:52:00,2.95,16448.0,United Kingdom
384,530244,71053,WHITE METAL LANTERN,7500,2025-12-12 02:17:00,3.39,15629.0,United Kingdom
485,530140,20725,LUNCH BAG RED RETROSPOT,9999,2025-11-07 19:50:00,1.65,17066.0,United Kingdom


3 rows have very large Quantity values (5000, 7500, 9999), much higher than every other order in the data. Each of these alone would add a huge amount to total revenue, which does not match the rest of the dataset. These look like data entry mistakes, not real orders, so we remove them.

In [44]:
df = df[df["Quantity"] <= 500]
df.shape

(493, 8)

Fixing inconsistent country names
Some country names appear in different letter cases (e.g. "GERMANY" and "Germany"), which would cause the same country to be treated as two separate groups during analysis. We will standardize all country names to proper case.

In [45]:
df["Country"] = df["Country"].str.title()
df["Country"].unique()

<ArrowStringArray>
['United Kingdom',           'Eire',        'Germany',       'Portugal',
          'Spain',      'Australia',        'Belgium',    'Switzerland',
         'France',    'Netherlands']
Length: 10, dtype: str

Saving the cleaned data

In [47]:
df.to_csv("../data/Processed/cleaned_sales.csv", index=False)

In [48]:
os.path.exists("../data/processed/cleaned_sales.csv")

True

## Part 3: Analysis

In [49]:
df["Year"] = df["InvoiceDate"].dt.year
df["Month"] = df["InvoiceDate"].dt.month
df["Day"] = df["InvoiceDate"].dt.day
df["Revenue"] = df["Quantity"] * df["UnitPrice"]

df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Year,Month,Day,Revenue
0,530331,22961,JUMBO BAG RED RETROSPOT,37,2025-08-28 17:08:00,2.08,17013.0,United Kingdom,2025,8,28,76.96
1,530359,22457,CREAM CUPID HEARTS COAT HANGER,51,2025-10-13 11:48:00,2.75,13181.0,United Kingdom,2025,10,13,140.25
2,530400,21212,PACK OF 72 RETROSPOT CAKE CASES,11,2025-12-18 16:59:00,0.55,17763.0,United Kingdom,2025,12,18,6.05
3,530024,22720,POPCORN HOLDER,33,2025-09-17 03:55:00,0.85,17122.0,United Kingdom,2025,9,17,28.05
5,530173,22961B,RABBIT NIGHT LIGHT,25,2025-07-30 01:20:00,2.08,18100.0,United Kingdom,2025,7,30,52.00


Question 1 — Total revenue

In [51]:
total_revenue = df["Revenue"].sum()
total_revenue

np.float64(53057.56)

The total revenue after cleaning is 53,057.56. This is lower than the raw dataset's total because we removed cancelled orders, invalid entries and clear data errors, so this number reflects real, valid sales only.

Question 2 — Number of transactions/orders

In [52]:
num_transactions = df.shape[0]
num_transactions

493

There are 493 valid transactions in the cleaned dataset

Question 3 — Average order value

In [53]:
average_order_value = df["Revenue"].mean()
average_order_value

np.float64(107.62182555780933)

The average order value is about 107.62. This tells us how much revenue a typical transaction brings in

Question 4 — Top products by revenue

In [54]:
top_products = (
    df.groupby("Description")["Revenue"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
)
top_products

Description
POSTAGE                               11466.00
REGENCY CAKESTAND 3 TIER               8555.25
CREAM CUPID HEARTS COAT HANGER         8159.25
SET 7 BABUSHKA NESTING BOXES           5610.00
PARTY BUNTING                          4315.20
RABBIT NIGHT LIGHT                     2477.28
WHITE METAL LANTERN                    2257.74
WHITE HANGING HEART T-LIGHT HOLDER     2008.95
JUMBO BAG RED RETROSPOT                1946.88
POPCORN HOLDER                         1459.45
Name: Revenue, dtype: float64

POSTAGE made the most money, followed by REGENCY CAKESTAND 3 TIER and CREAM CUPID HEARTS COAT HANGER. A few products bring in most of the revenue, so the business should keep a close eye on these top sellers.

Question 5 — Top countries by revenue

In [55]:
top_countries = (
    df.groupby("Country")["Revenue"]
    .sum()
    .sort_values(ascending=False)
    .head(5)
)
top_countries

Country
United Kingdom    40977.37
Australia          2569.07
Eire               1867.55
Germany            1578.71
Switzerland        1413.66
Name: Revenue, dtype: float64

The United Kingdom brings in far more revenue than any other country, followed by Australia, Eire, Germany and Switzerland. Most of the business comes from the UK.

Question 6 — Top customers by revenue

In [56]:
top_customers = (
    df[df["CustomerID"] != "Unknown"]
    .groupby("CustomerID")["Revenue"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
)
top_customers

CustomerID
15283.0    1044.0
15232.0     990.0
14356.0     954.0
14371.0     918.0
12807.0     882.0
16375.0     846.0
12994.0     840.6
12872.0     828.0
17138.0     828.0
18052.0     756.0
Name: Revenue, dtype: float64

Customer 15283 contributed the most revenue, followed by customers 15232 and 14356. A small group of customers bring in more revenue than others

Question 7 — Revenue trend over time

In [58]:
import calendar

monthly_revenue = df.groupby("Month")["Revenue"].sum()
monthly_revenue.index = [calendar.month_abbr[m] for m in monthly_revenue.index]
monthly_revenue

Jan    2893.59
Feb    4661.76
Mar    4854.87
Apr    3720.62
May    4220.46
Jun    3740.91
Jul    5658.30
Aug    5565.74
Sep    4356.41
Oct    4422.85
Nov    4938.90
Dec    4023.15
Name: Revenue, dtype: float64

July had the highest revenue, while January had the lowest. Revenue moves up and down across the months

Question 8 — One interesting pattern discovered

In [60]:
df["Quantity"].corr(df["Revenue"])

np.float64(0.42575081296393597)

Quantity and Revenue have a moderate positive relationship (0.43). This means transactions with higher quantity tend to have higher revenue, but the connection is not very strong, since price also plays a big role

## Personal Business Questions

Custom Question 1 — Which day of the week has the most transactions?

In [62]:
df["DayOfWeek"] = df["InvoiceDate"].dt.day_name()
transactions_by_day = df["DayOfWeek"].value_counts()
transactions_by_day

DayOfWeek
Saturday     79
Monday       75
Sunday       73
Thursday     70
Friday       69
Tuesday      67
Wednesday    60
Name: count, dtype: int64

Saturday has the most transactions, while Wednesday has the fewest. This could help the business plan staffing or run promotions on the busier days like Saturday and Monday.

Custom Question 2 — What is the average order value per country?

In [63]:
avg_order_by_country = (
    df.groupby("Country")["Revenue"]
    .mean()
    .sort_values(ascending=False)
)
avg_order_by_country

Country
France            303.637500
Australia         160.566875
Switzerland       128.514545
Germany           121.439231
Portugal          112.976667
Eire              109.855882
United Kingdom    104.801458
Belgium            78.246923
Netherlands        67.963333
Spain              66.140769
Name: Revenue, dtype: float64

France has the highest average order value, even though the United Kingdom brings in far more total revenue. This shows that UK sales come from many smaller orders, while countries like France have fewer but bigger orders. Looking at both total revenue and average order value gives a fuller picture of each country's buying pattern.